# 12 Multi Document Rag Chain

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `12-multi-document-rag-chain.ipynb`

In [1]:
# Start coding here
# ==================================================
# Notebook 12
# Multi Document RAG Chain
# ==================================================

import json
import pickle
import faiss
import pandas as pd
import numpy as np

from rank_bm25 import BM25Okapi

from transformers import pipeline

from sentence_transformers import SentenceTransformer, CrossEncoder

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

llm = pipeline("text2text-generation", model="google/flan-t5-base")

Device set to use cpu


In [3]:
children_df = pd.read_csv("artifacts/child_chunks.csv")

parents_df = pd.read_csv("artifacts/parent_chunks.csv")

metadata_df = pd.read_csv("artifacts/document_metadata.csv")

In [4]:
with open("artifacts/router_embeddings.pkl", "rb") as f:

    department_embeddings = pickle.load(f)

In [5]:
vector_indexes = {}

for dept in ["hr", "finance", "engineering"]:

    vector_indexes[dept] = faiss.read_index(f"artifacts/faiss/{dept}.index")

In [6]:
with open("artifacts/faiss/chunk_mapping.pkl", "rb") as f:

    chunk_mappings = pickle.load(f)

In [7]:
children_df = children_df.merge(
    metadata_df[["document_id", "department"]], on="document_id", how="left"
)

department_bm25 = {}
department_docs = {}

for dept in children_df["department"].unique():

    subset = children_df[children_df["department"] == dept].reset_index(drop=True)

    corpus = [text.lower().split() for text in subset["content"]]

    department_bm25[dept] = BM25Okapi(corpus)

    department_docs[dept] = subset

In [8]:
conversation_memory = []

In [9]:
def add_message(role, content):

    conversation_memory.append({"role": role, "content": content})

In [10]:
def build_history_context():

    text = ""

    for item in conversation_memory[-6:]:

        text += f"{item['role']}: " f"{item['content']}\n"

    return text

In [11]:
def rewrite_query(question):

    history = build_history_context()

    prompt = f"""
Rewrite the user question into a standalone question.

Conversation:

{history}

Question:

{question}

Standalone Question:
"""

    response = llm(prompt, max_new_tokens=64)

    return response[0]["generated_text"]

In [12]:
def route_query(query):

    query_embedding = embedding_model.encode(query)

    scores = {}

    for dept, dept_embedding in department_embeddings.items():

        score = cosine_similarity(
            query_embedding.reshape(1, -1), dept_embedding.reshape(1, -1)
        )[0][0]

        scores[dept] = float(score)

    return max(scores, key=scores.get)

In [13]:
def bm25_search(query, department, top_k=20):

    bm25 = department_bm25[department]

    docs = department_docs[department].copy()

    scores = bm25.get_scores(query.lower().split())

    docs["score"] = scores

    docs = docs.sort_values("score", ascending=False)

    return docs.head(top_k)

In [14]:
def bm25_search(query, department, top_k=20):

    bm25 = department_bm25[department]

    docs = department_docs[department].copy()

    scores = bm25.get_scores(query.lower().split())

    docs["score"] = scores

    docs = docs.sort_values("score", ascending=False)

    return docs.head(top_k)

In [15]:
def semantic_search(query, department, top_k=20):

    query_embedding = embedding_model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, ids = vector_indexes[department].search(query_embedding, top_k)

    mapping = chunk_mappings[department]

    results = []

    for score, idx in zip(scores[0], ids[0]):

        row = mapping.iloc[idx]

        results.append(
            {
                "parent_id": row["parent_id"],
                "content": row["content"],
                "score": float(score),
            }
        )

    return results

In [16]:
def hybrid_candidates(query):

    dept = route_query(query)

    bm25_results = bm25_search(query, dept)

    semantic_results = semantic_search(query, dept)

    candidates = {}

    for _, row in bm25_results.iterrows():

        candidates[row["parent_id"]] = row["content"]

    for row in semantic_results:

        candidates[row["parent_id"]] = row["content"]

    return list(candidates.items())

In [17]:
def rerank_documents(query, candidates, top_k=5):

    pairs = [[query, text] for _, text in candidates]

    scores = cross_encoder.predict(pairs)

    ranked = []

    for (doc_id, text), score in zip(candidates, scores):

        ranked.append({"parent_id": doc_id, "content": text, "score": float(score)})

    ranked = sorted(ranked, key=lambda x: x["score"], reverse=True)

    return ranked[:top_k]

In [18]:
parent_metadata = parents_df.merge(metadata_df, on="document_id", how="left")

In [19]:
def get_citation(parent_id):

    row = parent_metadata[parent_metadata["parent_id"] == parent_id]

    if len(row) == 0:
        return None

    row = row.iloc[0]

    return {"document": row["filename"], "department": row["department"]}

In [20]:
def build_context(ranked_docs):

    context = ""

    for doc in ranked_docs:

        citation = get_citation(doc["parent_id"])

        context += f"""
SOURCE:
{citation['document']}

CONTENT:
{doc['content']}

====================================

"""
    return context

In [21]:
def generate_answer(query, context):

    prompt = f"""
Answer ONLY from the context.

If not found say:

I could not find that information.

CONTEXT:

{context}

QUESTION:

{query}

ANSWER:
"""

    response = llm(prompt, max_new_tokens=256)

    return response[0]["generated_text"]

In [22]:
def ask_assistant(question):

    standalone_question = rewrite_query(question)

    candidates = hybrid_candidates(standalone_question)

    ranked_docs = rerank_documents(standalone_question, candidates)

    context = build_context(ranked_docs)

    answer = generate_answer(standalone_question, context)

    add_message("user", question)

    add_message("assistant", answer)

    return {
        "question": question,
        "standalone_question": standalone_question,
        "answer": answer,
        "citations": [get_citation(doc["parent_id"]) for doc in ranked_docs],
    }

In [23]:
response = ask_assistant("What was revenue in 2026?")

response

{'question': 'What was revenue in 2026?',
 'standalone_question': 'What was revenue in 2026?',
 'answer': '2.1B',
 'citations': [{'document': 'Annual_Report_2026.txt', 'department': 'finance'},
  {'document': 'Annual_Report_2026.txt', 'department': 'finance'}]}

In [24]:
response = ask_assistant("Did it increase?")

response

{'question': 'Did it increase?',
 'standalone_question': 'What was revenue in 2026?',
 'answer': '2.1B',
 'citations': [{'document': 'Annual_Report_2026.txt', 'department': 'finance'},
  {'document': 'Annual_Report_2026.txt', 'department': 'finance'}]}

In [25]:
for citation in response["citations"]:

    print(citation)

{'document': 'Annual_Report_2026.txt', 'department': 'finance'}
{'document': 'Annual_Report_2026.txt', 'department': 'finance'}
